# Genie Space Configuration — Sample Health Plans Member & Claims Analytics

This notebook is a **modular configuration tool** for managing the Genie space. Edit the configuration cells below, then run the execution cells to create or update the space.

## Notebook Structure
| Cell | Purpose | Action |
|---|---|---|
| **Cells 2-7** | Configuration (edit these) | Define space settings, instructions, metric views, questions, SQLs, benchmarks |
| **Cell 8** | Helper functions | Builds the `serialized_space` JSON from your config |
| **Cell 9** | Create / Update space | Calls the Genie API to apply changes |
| **Cell 10** | Validate space | Reads back the space and displays a summary |

## How to Use
1. Edit any configuration cell (Cells 2–7)
2. Run **Cell 8** (helpers), then **Cell 9** (apply), then **Cell 10** (validate)
3. To create a **new** space: set `SPACE_ID = None` in Cell 2
4. To **update** an existing space: set `SPACE_ID = "<existing_id>"` in Cell 2

In [0]:
# =============================================================================
# SPACE CONFIGURATION
# Edit these values to configure the Genie space.
# =============================================================================

SPACE_TITLE = "Sample Health Plans Member & Claims Analytics"

SPACE_DESCRIPTION = (
    "Sample Health Plans comprehensive analytics space covering member enrollment, "
    "claims financials, and cross-domain utilization. Powered by 5 metric views "
    "with 30 base KPIs and 22 window measures across Member, Claim, and "
    "Member-Claim domains. Includes industry benchmarks for PMPM, clean claim "
    "rate, utilization, churn, and PCP access."
)

# Set to None to CREATE a new space, or an existing ID to UPDATE
SPACE_ID = ""
WAREHOUSE_ID = "<your-sql-warehouse-id>"
PARENT_PATH = "/Users/your.name@company.com"

print(f"Space: {SPACE_TITLE}")
print(f"Mode:  {'UPDATE existing' if SPACE_ID else 'CREATE new'}")

Space: Sample Health Plans Member & Claims Analytics
Mode:  CREATE new


In [0]:
# =============================================================================
# GENERAL INSTRUCTIONS
# This is the single instruction block that teaches Genie about the space.
# Edit the text below to change how Genie interprets questions.
# The API allows exactly ONE text_instructions entry.
# =============================================================================

GENERAL_INSTRUCTIONS = """
This Genie space provides natural-language access to Sample Health Plans member enrollment, claims, and cross-domain utilization analytics. It is powered by 5 metric views covering 30 base KPIs and 22 window measures across 3 domains: Member, Claim, and Member-Claim.

LINES OF BUSINESS REFERENCE: COM = Commercial, MCR = Medicare, MCD = Medicaid, MMP = Medicare-Medicaid Plan, EXC = Exchange (ACA Marketplace). When users mention a line of business by full name, map it to the corresponding code for filtering.

INDUSTRY BENCHMARKS — Use these as context when answering questions or providing analysis:
- PMPM: Commercial $400-$600, Medicare $800-$1,200, Medicaid $300-$500
- Clean Claim Rate: Industry target >= 90%, top performers > 95%
- Utilization Rate: Typical 60-75% of members use services annually
- Churn Rate: Healthy plans target < 15% annually
- PCP Assigned Rate: Managed care target > 90%
- PCP Utilization Rate: Target > 50% for primary care access
- High-Cost Member Spend: Typically 5-10% of members drive 50-60% of spend
- Claims per 1,000 Members: Commercial 600-900/month, Medicare 1,000-1,500/month
- Inpatient Rate: Typically 5-15% of total claims

WINDOW MEASURES: Several metric views include window measures for temporal analysis. Rolling measures smooth volatility (Rolling 7d Claims, Rolling 3mo PMPM, Rolling 90d Clean Rate). Cumulative measures track YTD progress (YTD PMPM, Cumulative YTD Enrollments). Period-over-period measures compare current vs prior month (MoM Allowed Change %, MoM Active Member Growth %). Semiadditive measures provide point-in-time snapshots (Latest Member Count). Window logic is embedded in the metric view — query them with the same MEASURE() syntax as base KPIs.

METRIC VIEW SELECTION GUIDE:
- For membership demographics and counts: use mv_membership
- For enrollment/termination lifecycle trends: use mv_member_lifecycle
- For claims volume, financial amounts, and operational quality: use mv_claims
- For per-member-per-month cost and utilization rates: use mv_utilization
- For member-level cost profiles and high-cost member analysis: use mv_member_cost_profile
""".strip()

print(f"Instructions length: {len(GENERAL_INSTRUCTIONS)} chars")
print(f"Preview: {GENERAL_INSTRUCTIONS[:100]}...")

Instructions length: 2124 chars
Preview: This Genie space provides natural-language access to Sample Health Plans member enrollment, claims, and c...


In [0]:
# =============================================================================
# METRIC VIEW DESCRIPTIONS
# Map fully-qualified metric view names to their descriptions.
# Keys MUST be sorted alphabetically (the helper function enforces this).
# =============================================================================

METRIC_VIEW_DESCRIPTIONS = {
    "aira_test.aibi_workshop_schema.mv_claims": (
        "Comprehensive claims financial and operational metrics. "
        "15 base measures (C1-C10, MC5, MC10) + 8 window measures (W5-W8). "
        "Source: fact_claim_detail JOIN fact_claim_header."
    ),
    "aira_test.aibi_workshop_schema.mv_member_cost_profile": (
        "Member-level cost and access profile. "
        "9 measures: Avg Cost/Member (MC4), High-Cost Count/Spend% (MC7-MC8), "
        "PCP Util Rate (MC9). Source: vw_member_claims_profile."
    ),
    "aira_test.aibi_workshop_schema.mv_member_lifecycle": (
        "Member enrollment lifecycle. "
        "3 base (M3-M5) + 2 window measures (W1-W2). "
        "Source: dim_member_history."
    ),
    "aira_test.aibi_workshop_schema.mv_membership": (
        "Member demographic snapshot. "
        "7 measures: Active Members (M1), Subscribers (M2), PCP Rate (M6), "
        "Medicaid (M7), LOB/Geo (M8-M9), Deceased (M10). Source: dim_member."
    ),
    "aira_test.aibi_workshop_schema.mv_utilization": (
        "Cross-domain utilization at member-month grain. "
        "8 base (MC1-MC3, MC5-MC6) + 12 window measures (W3-W4, W9-W13). "
        "Source: vw_member_month_utilization."
    ),
}

print(f"Metric views configured: {len(METRIC_VIEW_DESCRIPTIONS)}")
for name in METRIC_VIEW_DESCRIPTIONS:
    print(f"  • {name.split('.')[-1]}")

# Validate alphabetical order
keys = list(METRIC_VIEW_DESCRIPTIONS.keys())
assert keys == sorted(keys), "ERROR: Metric view keys must be sorted alphabetically!"
print("\n✅ Keys are sorted alphabetically")

Metric views configured: 5
  • mv_claims
  • mv_member_cost_profile
  • mv_member_lifecycle
  • mv_membership
  • mv_utilization

✅ Keys are sorted alphabetically


In [0]:
# =============================================================================
# SAMPLE QUESTIONS
# These appear in the Genie chat UI as suggested questions.
# Add, remove, or edit questions as needed.
# =============================================================================

SAMPLE_QUESTIONS = [
    # --- Membership ---
    "How many active members do we have by line of business?",
    "Show me active member distribution by state",
    "What is the subscriber to member ratio by line of business?",
    "How many deceased members do we have and what is their distribution?",
    # --- Lifecycle ---
    "What is the churn rate by LOB?",
    "What are the cumulative year-to-date enrollments?",
    # --- Claims ---
    "Show me the clean claim rate by line of business for the last 3 months",
    "How many claims and claim lines do we process monthly?",
    "What is the average paid per claim by claim type (Professional vs Institutional)?",
    "What percentage of claims are inpatient by line of business?",
    "Show me the billed vs allowed vs paid amounts by line of business",
    "Show me claims breakdown by benefit category",
    "What are the top 10 procedure codes by total paid amount?",
    "Show me the month-over-month allowed amount change by LOB",
    "What is the rolling 90-day clean claim rate trend?",
    # --- Utilization ---
    "What is the PMPM trend by month for Commercial members?",
    "How does our rolling 3-month PMPM compare across lines of business?",
    "What is the year-to-date PMPM for Medicare members?",
    "Show me month-over-month active member growth trend",
    "Compare Medicaid vs Medicare PMPM and utilization rates",
    # --- Cost Profile ---
    "What percentage of our members are high-cost and how much spend do they drive?",
    "What is the PCP utilization rate by line of business?",
]

print(f"Sample questions configured: {len(SAMPLE_QUESTIONS)}")

Sample questions configured: 22


In [0]:
# =============================================================================
# EXAMPLE QUESTION SQLS
# These teach Genie HOW to answer questions (part of instructions).
# Each tuple is (question_text, sql_query).
# Add, remove, or edit pairs as needed.
# =============================================================================

C = "aira_test.aibi_workshop_schema"  # shorthand for readability

EXAMPLE_QUESTION_SQLS = [
    # --- Membership (mv_membership) ---
    (
        "How many active members do we have by line of business?",
        f"SELECT `Line of Business`, `Line of Business Name`, MEASURE(`Active Members`) AS active_members, MEASURE(`Active Subscribers`) AS active_subscribers FROM {C}.mv_membership GROUP BY ALL ORDER BY active_members DESC"
    ),
    (
        "Show me active member distribution by state",
        f"SELECT `State`, MEASURE(`Active Members`) AS active_members, MEASURE(`PCP Assigned Rate`) AS pcp_assigned_rate FROM {C}.mv_membership GROUP BY ALL ORDER BY active_members DESC"
    ),
    (
        "What is the subscriber to member ratio by line of business?",
        f"SELECT `Line of Business`, `Line of Business Name`, MEASURE(`Active Members`) AS active_members, MEASURE(`Active Subscribers`) AS active_subscribers, CAST(MEASURE(`Active Members`) AS DOUBLE) / NULLIF(MEASURE(`Active Subscribers`), 0) AS members_per_subscriber FROM {C}.mv_membership GROUP BY ALL ORDER BY members_per_subscriber DESC"
    ),
    (
        "How many deceased members do we have and what is their distribution?",
        f"SELECT `Line of Business`, `Line of Business Name`, MEASURE(`Deceased Member Count`) AS deceased_count, MEASURE(`Active Members`) AS active_members FROM {C}.mv_membership GROUP BY ALL ORDER BY deceased_count DESC"
    ),
    # --- Lifecycle (mv_member_lifecycle) ---
    (
        "What is the churn rate by LOB?",
        f"SELECT `Line of Business`, MEASURE(`New Member Enrollment`) AS new_enrollments, MEASURE(`Member Terminations`) AS terminations, MEASURE(`Member Churn Rate`) AS churn_rate FROM {C}.mv_member_lifecycle GROUP BY ALL ORDER BY churn_rate DESC"
    ),
    (
        "What are the cumulative year-to-date enrollments?",
        f"SELECT `First Enrollment Month`, `Line of Business`, MEASURE(`New Member Enrollment`) AS monthly_enrollments, MEASURE(`Cumulative YTD New Enrollments`) AS ytd_enrollments FROM {C}.mv_member_lifecycle WHERE `First Enrollment Month` >= DATE'2024-01-01' GROUP BY ALL ORDER BY `First Enrollment Month`, `Line of Business`"
    ),
    # --- Claims (mv_claims) ---
    (
        "Show me the clean claim rate by line of business for the last 3 months",
        f"SELECT `Service Month`, `Line of Business`, MEASURE(`Clean Claim Rate`) AS clean_claim_rate, MEASURE(`Total Claim Lines`) AS total_lines FROM {C}.mv_claims WHERE `Service Month` >= DATE_TRUNC('MONTH', CURRENT_DATE - INTERVAL 3 MONTH) GROUP BY ALL ORDER BY `Service Month`, clean_claim_rate DESC"
    ),
    (
        "How many claims and claim lines do we process monthly?",
        f"SELECT `Service Month`, MEASURE(`Total Claims`) AS total_claims, MEASURE(`Total Claim Lines`) AS total_lines, MEASURE(`Lines per Claim`) AS lines_per_claim, MEASURE(`Total Paid Amount`) AS total_paid FROM {C}.mv_claims GROUP BY ALL ORDER BY `Service Month` DESC LIMIT 12"
    ),
    (
        "What is the average paid per claim by claim type (Professional vs Institutional)?",
        f"SELECT `Claim Type`, MEASURE(`Total Claims`) AS total_claims, MEASURE(`Total Paid Amount`) AS total_paid, MEASURE(`Avg Paid per Claim`) AS avg_paid_per_claim, MEASURE(`Avg Paid per Claim Line`) AS avg_paid_per_line FROM {C}.mv_claims GROUP BY ALL ORDER BY `Claim Type`"
    ),
    (
        "What percentage of claims are inpatient by line of business?",
        f"SELECT `Line of Business`, MEASURE(`Total Claims`) AS total_claims, MEASURE(`Inpatient Claim Count`) AS inpatient_claims, CAST(MEASURE(`Inpatient Claim Count`) AS DOUBLE) / NULLIF(MEASURE(`Total Claims`), 0) AS inpatient_pct FROM {C}.mv_claims GROUP BY ALL ORDER BY inpatient_pct DESC"
    ),
    (
        "Show me the billed vs allowed vs paid amounts by line of business",
        f"SELECT `Line of Business`, MEASURE(`Total Billed Amount`) AS total_billed, MEASURE(`Total Allowed Amount`) AS total_allowed, MEASURE(`Total Paid Amount`) AS total_paid, MEASURE(`Total Paid Amount`) / NULLIF(MEASURE(`Total Billed Amount`), 0) AS paid_to_billed_ratio, MEASURE(`Total Paid Amount`) / NULLIF(MEASURE(`Total Allowed Amount`), 0) AS paid_to_allowed_ratio FROM {C}.mv_claims GROUP BY ALL ORDER BY total_paid DESC"
    ),
    (
        "Show me claims breakdown by benefit category",
        f"SELECT `Benefit Category`, MEASURE(`Total Claims`) AS total_claims, MEASURE(`Total Claim Lines`) AS total_lines, MEASURE(`Total Paid Amount`) AS total_paid, MEASURE(`Avg Paid per Claim Line`) AS avg_paid_per_line FROM {C}.mv_claims GROUP BY ALL ORDER BY total_paid DESC"
    ),
    (
        "What are the top 10 procedure codes by total paid amount?",
        f"WITH ranked AS (SELECT `Procedure Code`, MEASURE(`Total Claim Lines`) AS total_lines, MEASURE(`Total Paid Amount`) AS total_paid, MEASURE(`Avg Paid per Claim Line`) AS avg_per_line, ROW_NUMBER() OVER (ORDER BY MEASURE(`Total Paid Amount`) DESC) AS rn FROM {C}.mv_claims GROUP BY ALL) SELECT `Procedure Code`, total_lines, total_paid, avg_per_line FROM ranked WHERE rn <= 10 ORDER BY total_paid DESC"
    ),
    (
        "Show me the month-over-month allowed amount change by LOB",
        f"SELECT `Service Month`, `Line of Business`, MEASURE(`Current Month Allowed`) AS current_allowed, MEASURE(`Prior Month Allowed`) AS prior_allowed, MEASURE(`MoM Allowed Change Pct`) AS mom_change_pct FROM {C}.mv_claims WHERE `Service Month` >= DATE'2025-01-01' GROUP BY ALL ORDER BY `Service Month`, `Line of Business`"
    ),
    (
        "What is the rolling 90-day clean claim rate trend?",
        f"SELECT `Service Month`, `Line of Business`, MEASURE(`Clean Claim Rate`) AS point_clean_rate, MEASURE(`Rolling 90d Clean Rate`) AS rolling_90d_clean_rate FROM {C}.mv_claims WHERE `Service Month` >= DATE'2025-01-01' GROUP BY ALL ORDER BY `Service Month`, `Line of Business`"
    ),
    # --- Utilization (mv_utilization) ---
    (
        "What is the PMPM trend by month for Commercial members?",
        f"SELECT `Enrollment Month`, MEASURE(`PMPM`) AS pmpm, MEASURE(`Member Months`) AS member_months, MEASURE(`Total Paid Amount`) AS total_paid FROM {C}.mv_utilization WHERE `Line of Business` = 'COM' GROUP BY ALL ORDER BY `Enrollment Month`"
    ),
    (
        "How does our rolling 3-month PMPM compare across lines of business?",
        f"SELECT `Enrollment Month`, `Line of Business`, MEASURE(`PMPM`) AS current_pmpm, MEASURE(`Rolling 3mo PMPM`) AS rolling_3mo_pmpm FROM {C}.mv_utilization WHERE `Enrollment Month` >= DATE'2025-01-01' GROUP BY ALL ORDER BY `Enrollment Month`, `Line of Business`"
    ),
    (
        "What is the year-to-date PMPM for Medicare members?",
        f"SELECT `Enrollment Month`, MEASURE(`PMPM`) AS monthly_pmpm, MEASURE(`YTD PMPM`) AS ytd_pmpm, MEASURE(`YTD Total Paid`) AS ytd_paid, MEASURE(`YTD Member Months`) AS ytd_member_months FROM {C}.mv_utilization WHERE `Line of Business` = 'MCR' GROUP BY ALL ORDER BY `Enrollment Month`"
    ),
    (
        "Show me month-over-month active member growth trend",
        f"SELECT `Enrollment Month`, `Line of Business`, MEASURE(`Current Month Active Members`) AS current_members, MEASURE(`Prior Month Active Members`) AS prior_members, MEASURE(`MoM Active Member Growth Pct`) AS mom_growth_pct FROM {C}.mv_utilization WHERE `Enrollment Month` >= DATE'2025-01-01' GROUP BY ALL ORDER BY `Enrollment Month`, `Line of Business`"
    ),
    (
        "Compare Medicaid vs Medicare PMPM and utilization rates",
        f"SELECT `Enrollment Month`, `Line of Business`, MEASURE(`PMPM`) AS pmpm, MEASURE(`Utilization Rate`) AS utilization_rate, MEASURE(`Members with Claims`) AS members_with_claims, MEASURE(`Active Members`) AS active_members FROM {C}.mv_utilization WHERE `Line of Business` IN ('MCD', 'MCR') AND `Enrollment Month` >= DATE'2025-06-01' GROUP BY ALL ORDER BY `Enrollment Month`, `Line of Business`"
    ),
    # --- Cost Profile (mv_member_cost_profile) ---
    (
        "What percentage of our members are high-cost and how much spend do they drive?",
        f"SELECT `Line of Business`, MEASURE(`Active Members`) AS active_members, MEASURE(`High-Cost Member Count`) AS high_cost_count, MEASURE(`High-Cost Member Spend Pct`) AS high_cost_spend_pct, MEASURE(`Total Paid Amount`) AS total_paid FROM {C}.mv_member_cost_profile GROUP BY ALL ORDER BY high_cost_spend_pct DESC"
    ),
    (
        "What is the PCP utilization rate by line of business?",
        f"SELECT `Line of Business`, MEASURE(`Active Members`) AS active_members, MEASURE(`Members with PCP Visit`) AS members_with_pcp, MEASURE(`PCP Utilization Rate`) AS pcp_util_rate FROM {C}.mv_member_cost_profile GROUP BY ALL ORDER BY pcp_util_rate DESC"
    ),
]

print(f"Example question SQLs configured: {len(EXAMPLE_QUESTION_SQLS)}")

Example question SQLs configured: 22


In [0]:
# =============================================================================
# BENCHMARK QUESTIONS
# These EVALUATE Genie accuracy (separate from instructions).
# Each tuple is (question_text, expected_sql_answer).
# Genie does NOT learn from benchmarks — they are for testing only.
# =============================================================================

BENCHMARK_QUESTIONS = [
    # --- Membership (mv_membership) ---
    (
        "How many active members do we have by line of business?",
        f"SELECT `Line of Business`, `Line of Business Name`, MEASURE(`Active Members`) AS active_members, MEASURE(`Active Subscribers`) AS active_subscribers FROM {C}.mv_membership GROUP BY ALL ORDER BY active_members DESC"
    ),
    (
        "What is the PCP assigned rate for each line of business?",
        f"SELECT `Line of Business`, `Line of Business Name`, MEASURE(`Active Members`) AS active_members, MEASURE(`PCP Assigned Members`) AS pcp_assigned, MEASURE(`PCP Assigned Rate`) AS pcp_assigned_rate FROM {C}.mv_membership GROUP BY ALL ORDER BY pcp_assigned_rate DESC"
    ),
    (
        "How many Medicaid members do we have?",
        f"SELECT `Line of Business`, `Line of Business Name`, MEASURE(`Medicaid Member Count`) AS medicaid_count, MEASURE(`Active Members`) AS active_members FROM {C}.mv_membership WHERE `Line of Business` = 'MCD' GROUP BY ALL"
    ),
    (
        "Show active member count by state",
        f"SELECT `State`, MEASURE(`Active Members`) AS active_members FROM {C}.mv_membership GROUP BY ALL ORDER BY active_members DESC"
    ),
    # --- Lifecycle (mv_member_lifecycle) ---
    (
        "What is the member churn rate by line of business?",
        f"SELECT `Line of Business`, MEASURE(`New Member Enrollment`) AS new_enrollments, MEASURE(`Member Terminations`) AS terminations, MEASURE(`Member Churn Rate`) AS churn_rate FROM {C}.mv_member_lifecycle GROUP BY ALL ORDER BY churn_rate DESC"
    ),
    (
        "Show cumulative year-to-date new enrollments",
        f"SELECT `First Enrollment Month`, MEASURE(`New Member Enrollment`) AS monthly_enrollments, MEASURE(`Cumulative YTD New Enrollments`) AS ytd_enrollments FROM {C}.mv_member_lifecycle GROUP BY ALL ORDER BY `First Enrollment Month`"
    ),
    # --- Claims (mv_claims) ---
    (
        "What is the total paid amount by line of business this year?",
        f"SELECT `Line of Business`, MEASURE(`Total Claims`) AS total_claims, MEASURE(`Total Paid Amount`) AS total_paid, MEASURE(`Avg Paid per Claim`) AS avg_per_claim FROM {C}.mv_claims WHERE `Service Year` = 2026 GROUP BY ALL ORDER BY total_paid DESC"
    ),
    (
        "Show me billed vs allowed vs paid amounts by LOB",
        f"SELECT `Line of Business`, MEASURE(`Total Billed Amount`) AS billed, MEASURE(`Total Allowed Amount`) AS allowed, MEASURE(`Total Paid Amount`) AS paid FROM {C}.mv_claims GROUP BY ALL ORDER BY paid DESC"
    ),
    (
        "What is the clean claim rate by line of business?",
        f"SELECT `Line of Business`, MEASURE(`Total Claim Lines`) AS total_lines, MEASURE(`Clean Claims`) AS clean_claims, MEASURE(`Clean Claim Rate`) AS clean_rate FROM {C}.mv_claims GROUP BY ALL ORDER BY clean_rate DESC"
    ),
    (
        "Show me average paid per claim by claim type",
        f"SELECT `Claim Type`, MEASURE(`Total Claims`) AS total_claims, MEASURE(`Total Paid Amount`) AS total_paid, MEASURE(`Avg Paid per Claim`) AS avg_paid FROM {C}.mv_claims GROUP BY ALL ORDER BY `Claim Type`"
    ),
    (
        "What percentage of claims are inpatient?",
        f"SELECT `Line of Business`, MEASURE(`Total Claims`) AS total_claims, MEASURE(`Inpatient Claim Count`) AS inpatient, CAST(MEASURE(`Inpatient Claim Count`) AS DOUBLE) / NULLIF(MEASURE(`Total Claims`), 0) AS inpatient_pct FROM {C}.mv_claims GROUP BY ALL ORDER BY inpatient_pct DESC"
    ),
    (
        "What is the rolling 90-day clean claim rate by LOB?",
        f"SELECT `Service Month`, `Line of Business`, MEASURE(`Clean Claim Rate`) AS point_rate, MEASURE(`Rolling 90d Clean Rate`) AS rolling_90d_rate FROM {C}.mv_claims WHERE `Service Month` >= DATE'2025-06-01' GROUP BY ALL ORDER BY `Service Month`, `Line of Business`"
    ),
    (
        "Show month-over-month allowed amount change",
        f"SELECT `Service Month`, `Line of Business`, MEASURE(`Current Month Allowed`) AS current_allowed, MEASURE(`Prior Month Allowed`) AS prior_allowed, MEASURE(`MoM Allowed Change Pct`) AS mom_pct FROM {C}.mv_claims WHERE `Service Month` >= DATE'2025-06-01' GROUP BY ALL ORDER BY `Service Month`, `Line of Business`"
    ),
    # --- Utilization (mv_utilization) ---
    (
        "What is the PMPM by line of business?",
        f"SELECT `Line of Business`, MEASURE(`Member Months`) AS member_months, MEASURE(`Total Paid Amount`) AS total_paid, MEASURE(`PMPM`) AS pmpm FROM {C}.mv_utilization GROUP BY ALL ORDER BY pmpm DESC"
    ),
    (
        "Show PMPM trend for Commercial members",
        f"SELECT `Enrollment Month`, MEASURE(`PMPM`) AS pmpm, MEASURE(`Member Months`) AS member_months FROM {C}.mv_utilization WHERE `Line of Business` = 'COM' GROUP BY ALL ORDER BY `Enrollment Month`"
    ),
    (
        "What is the rolling 3-month PMPM by LOB?",
        f"SELECT `Enrollment Month`, `Line of Business`, MEASURE(`PMPM`) AS monthly_pmpm, MEASURE(`Rolling 3mo PMPM`) AS rolling_3mo_pmpm FROM {C}.mv_utilization WHERE `Enrollment Month` >= DATE'2025-06-01' GROUP BY ALL ORDER BY `Enrollment Month`, `Line of Business`"
    ),
    (
        "What is the YTD PMPM for Medicare?",
        f"SELECT `Enrollment Month`, MEASURE(`PMPM`) AS monthly_pmpm, MEASURE(`YTD PMPM`) AS ytd_pmpm, MEASURE(`YTD Member Months`) AS ytd_mm FROM {C}.mv_utilization WHERE `Line of Business` = 'MCR' AND `Enrollment Year` = 2026 GROUP BY ALL ORDER BY `Enrollment Month`"
    ),
    (
        "What is the utilization rate by line of business?",
        f"SELECT `Line of Business`, MEASURE(`Active Members`) AS active, MEASURE(`Members with Claims`) AS with_claims, MEASURE(`Utilization Rate`) AS util_rate FROM {C}.mv_utilization GROUP BY ALL ORDER BY util_rate DESC"
    ),
    (
        "Show month-over-month active member growth",
        f"SELECT `Enrollment Month`, `Line of Business`, MEASURE(`Current Month Active Members`) AS current, MEASURE(`Prior Month Active Members`) AS prior, MEASURE(`MoM Active Member Growth Pct`) AS growth_pct FROM {C}.mv_utilization WHERE `Enrollment Month` >= DATE'2025-06-01' GROUP BY ALL ORDER BY `Enrollment Month`, `Line of Business`"
    ),
    # --- Cost Profile (mv_member_cost_profile) ---
    (
        "What percentage of spend is driven by high-cost members?",
        f"SELECT `Line of Business`, MEASURE(`Active Members`) AS active, MEASURE(`High-Cost Member Count`) AS high_cost_count, MEASURE(`High-Cost Member Spend Pct`) AS high_cost_spend_pct FROM {C}.mv_member_cost_profile GROUP BY ALL ORDER BY high_cost_spend_pct DESC"
    ),
    (
        "What is the PCP utilization rate by LOB?",
        f"SELECT `Line of Business`, MEASURE(`Active Members`) AS active, MEASURE(`Members with PCP Visit`) AS pcp_visits, MEASURE(`PCP Utilization Rate`) AS pcp_rate FROM {C}.mv_member_cost_profile GROUP BY ALL ORDER BY pcp_rate DESC"
    ),
    (
        "What is the average cost per member by line of business?",
        f"SELECT `Line of Business`, MEASURE(`Members with Claims`) AS members_w_claims, MEASURE(`Total Paid Amount`) AS total_paid, MEASURE(`Avg Cost per Member`) AS avg_cost FROM {C}.mv_member_cost_profile GROUP BY ALL ORDER BY avg_cost DESC"
    ),
]

print(f"Benchmark questions configured: {len(BENCHMARK_QUESTIONS)}")

Benchmark questions configured: 22


In [0]:
# =============================================================================
# HELPER FUNCTIONS
# Run this cell before Cell 9 (Create/Update) or Cell 10 (Validate).
# =============================================================================

import json
import uuid
import requests


def get_workspace_url():
    """Get the current workspace URL."""
    return spark.conf.get("spark.databricks.workspaceUrl")


def get_api_headers():
    """Get authorization headers for the Genie API."""
    token = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().apiToken().get()
    )
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }


def _sorted_hex_ids(n: int) -> list[str]:
    """Generate *n* sorted 32-char lowercase hex UUIDs."""
    return sorted(uuid.uuid4().hex for _ in range(n))


def build_serialized_space(
    general_instructions: str,
    metric_view_descriptions: dict[str, str],
    sample_questions: list[str],
    example_question_sqls: list[tuple[str, str]],
    benchmark_questions: list[tuple[str, str]],
) -> str:
    """
    Build the serialized_space JSON string from user-editable config.

    Returns a JSON string ready for the Genie Space API.
    All IDs are auto-generated, sorted, and guaranteed unique.
    """

    # ---- Generate sorted IDs for every section ----
    sq_ids = _sorted_hex_ids(len(sample_questions))
    eq_ids = _sorted_hex_ids(len(example_question_sqls))
    bm_ids = _sorted_hex_ids(len(benchmark_questions))
    ti_id  = uuid.uuid4().hex                           # single instruction

    # Verify global uniqueness
    all_ids = sq_ids + eq_ids + bm_ids + [ti_id]
    assert len(all_ids) == len(set(all_ids)), "UUID collision — rerun the cell."

    # ---- config.sample_questions ----
    config_sq = [
        {"id": sq_ids[i], "question": [q]}
        for i, q in enumerate(sample_questions)
    ]

    # ---- data_sources.metric_views (sorted by identifier) ----
    mv_list = [
        {"identifier": k, "description": [v]}
        for k, v in sorted(metric_view_descriptions.items())
    ]

    # ---- instructions.text_instructions (exactly one entry) ----
    text_instr = [{"id": ti_id, "content": [general_instructions]}]

    # ---- instructions.example_question_sqls (sorted by id) ----
    ex_sqls = [
        {"id": eq_ids[i], "question": [q], "sql": [sql]}
        for i, (q, sql) in enumerate(example_question_sqls)
    ]

    # ---- benchmarks.questions (sorted by id) ----
    bm_list = [
        {
            "id": bm_ids[i],
            "question": [q],
            "answer": [{"format": "SQL", "content": [sql]}],
        }
        for i, (q, sql) in enumerate(benchmark_questions)
    ]

    # ---- Assemble full structure ----
    payload = {
        "version": 2,
        "config": {"sample_questions": config_sq},
        "data_sources": {"metric_views": mv_list},
        "instructions": {
            "text_instructions": text_instr,
            "example_question_sqls": ex_sqls,
        },
        "benchmarks": {"questions": bm_list},
    }

    return json.dumps(payload)


print("✅ Helper functions loaded: get_workspace_url, get_api_headers, build_serialized_space")

✅ Helper functions loaded: get_workspace_url, get_api_headers, build_serialized_space


In [0]:
# =============================================================================
# CREATE OR UPDATE GENIE SPACE
# Run this cell to apply the configuration from Cells 2–7.
# =============================================================================

serialised = build_serialized_space(
    general_instructions=GENERAL_INSTRUCTIONS,
    metric_view_descriptions=METRIC_VIEW_DESCRIPTIONS,
    sample_questions=SAMPLE_QUESTIONS,
    example_question_sqls=EXAMPLE_QUESTION_SQLS,
    benchmark_questions=BENCHMARK_QUESTIONS,
)

ws_url  = get_workspace_url()
headers = get_api_headers()

if SPACE_ID:
    # ---------- UPDATE existing space ----------
    print(f"Updating space {SPACE_ID} ...")
    resp = requests.patch(
        f"https://{ws_url}/api/2.0/genie/spaces/{SPACE_ID}",
        headers=headers,
        json={
            "title": SPACE_TITLE,
            "description": SPACE_DESCRIPTION,
            "serialized_space": serialised,
        },
    )
else:
    # ---------- CREATE new space ----------
    print("Creating new space ...")
    resp = requests.post(
        f"https://{ws_url}/api/2.0/genie/spaces",
        headers=headers,
        json={
            "title": SPACE_TITLE,
            "description": SPACE_DESCRIPTION,
            "warehouse_id": WAREHOUSE_ID,
            "parent_path": PARENT_PATH,
            "serialized_space": serialised,
        },
    )

# ---------- Report result ----------
if resp.status_code == 200:
    result = resp.json()
    new_id = result.get("space_id", SPACE_ID)
    print(f"\n✅ SUCCESS")
    print(f"   Space ID   : {new_id}")
    print(f"   Title       : {result.get('title')}")
    print(f"   Description : {result.get('description', '')[:120]}...")
    if not SPACE_ID:
        print(f"\n⚠️  Copy this ID into Cell 2 for future updates:")
        print(f'   SPACE_ID = "{new_id}"')
else:
    print(f"\n❌ FAILED ({resp.status_code})")
    err = resp.json()
    print(f"   Error : {err.get('error_code')}")
    print(f"   Message: {err.get('message', '')[:300]}")

Creating new space ...

✅ SUCCESS
   Space ID   : 01f1311fc06b1f619c9746ead3eab8f0
   Title       : Sample Health Plans Member & Claims Analytics
   Description : Sample Health Plans comprehensive analytics space covering member enrollment, claims financials, and cross-domain utilization...

⚠️  Copy this ID into Cell 2 for future updates:
   SPACE_ID = "01f1311fc06b1f619c9746ead3eab8f0"


In [0]:
# =============================================================================
# VALIDATE SPACE
# Read back the space config and display a summary.
# =============================================================================

target_id = SPACE_ID
if not target_id:
    print("No SPACE_ID set — set it in Cell 2 after creating a space.")
else:
    ws_url  = get_workspace_url()
    headers = get_api_headers()

    resp = requests.get(
        f"https://{ws_url}/api/2.0/genie/spaces/{target_id}"
        "?include_serialized_space=true",
        headers=headers,
    )

    if resp.status_code != 200:
        print(f"❌ Failed to read space: {resp.status_code}")
    else:
        data = resp.json()
        ss   = json.loads(data["serialized_space"])

        sqs  = ss.get("config", {}).get("sample_questions", [])
        mvs  = ss.get("data_sources", {}).get("metric_views", [])
        tis  = ss.get("instructions", {}).get("text_instructions", [])
        eqs  = ss.get("instructions", {}).get("example_question_sqls", [])
        bms  = ss.get("benchmarks", {}).get("questions", [])

        print("=" * 60)
        print("GENIE SPACE VALIDATION REPORT")
        print("=" * 60)
        print(f"Title        : {data.get('title')}")
        print(f"Space ID     : {data.get('space_id', target_id)}")
        print(f"Warehouse    : {data.get('warehouse_id', 'N/A')}")
        print(f"Description  : {data.get('description', '')[:120]}...")
        print()
        print(f"Metric Views        : {len(mvs)}")
        for mv in mvs:
            print(f"   • {mv['identifier'].split('.')[-1]}")
        print(f"Sample Questions    : {len(sqs)}")
        print(f"Example SQLs        : {len(eqs)}")
        print(f"Benchmark Questions : {len(bms)}")
        print(f"Text Instructions   : {len(tis)} block(s), "
              f"{sum(len(t['content'][0]) for t in tis)} chars")
        print()

        if bms:
            print("-" * 60)
            print("BENCHMARK QUESTIONS")
            print("-" * 60)
            for i, bm in enumerate(bms, 1):
                q = bm['question'][0]
                has_sql = bool(bm.get('answer'))
                marker = "✅" if has_sql else "⚠️"
                print(f"  {i:2d}. {marker} {q}")

        print()
        print("✅ Validation complete")

No SPACE_ID set — set it in Cell 2 after creating a space.
